# HuBMAP – Hacking the Human Vasculature: Mask R-CNN v2 (cross-patient robustness)

**Why v2.** v1 scored 0.644 AP@0.6 on held-out *tiles of slides it had trained on*, but only about 0.46 (public) / 0.55 (private) on the hidden test, which comes from *other patients*. The bottleneck is generalisation across patients (stain, scanner, tissue prep), not model capacity. v2 attacks that gap.

| Change | Why | Evidence |
|---|---|---|
| **HED stain jitter** (tissue only, calibrated strength) | simulates patient/lab stain differences | an ablation on this competition's data (slide-wise split, small U-Net) found HED jitter gave about +14 % relative Dice over plain augmentation; Macenko normalisation alone did not help |
| **Colour transfer from dataset-3 tiles** | dataset 3 = 9 *other* patients; only their colours are used, never labels | same motivation; cheap |
| **Whole-slide validation** (`SPLIT_MODE="wsi"`) | v1 validated on the same slides it trained on, so it overstated test performance | the 3rd-place write-up also noted a large CV-vs-LB gap |
| **Input size 1280** (was 1024) | the 3rd-place solution used 1400–2048 px | resolution consistently helped top solutions |
| **Multi-scale TTA**, measured on held-out data before use | cheap-to-test inference gain | chosen only if it beats flips-TTA by ≥ 0.005 AP |
| **All-data models** (`FULL_SEEDS`) | held-out folds leave out a slide's worth of clean data | final ensemble should use all labelled data |

Kept from v1: Mask R-CNN R50-FPN v2 with COCO weights, two-stage training (dataset 1+2, then clean dataset 1; the 3rd-place "multi-stage" trick), 2× mask loss, EMA, early stopping, flip TTA with instance-level merging.

**Be realistic.** None of these changes has been measured on your data yet. Expect gains of a few points, not a jump to 0.9. The printed held-out AP is a cross-patient estimate, which will look *lower* than v1's 0.644 because it is a harder test; compare it to the leaderboard, not to v1's CV.

**How to use (two steps)**
1. *Diagnose:* `FOLDS_TO_RUN = [0]` (then `[1]` in a copy). Read the slide table printed in section 3, the held-out AP, and the "settings to copy" lines at the end.
2. *Final models:* `FOLDS_TO_RUN = []`, `FULL_SEEDS = [0, 1, 2]` (one per notebook run if you like). Save the `fold*_best.pth` files and use the **lean inference notebook** with the settings printed in step 1. Don't mix v2 weights with v1 weights in one ensemble: the input size differs.

## 1. Setup and configuration
Everything you may want to change is in `CFG`. Use `DEBUG = True` first (one epoch on ~120 tiles at 512 px) to confirm the pipeline runs end to end.

In [ ]:
import os, gc, json, math, random, time, warnings, base64, zlib, copy, glob, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib as mpl
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


class CFG:
    # ---- run mode -------------------------------------------------------
    DEBUG = False            # True = tiny smoke test (few images, 1 epoch, small input)
    TRAIN = True             # False = inference only (load weights from WEIGHTS_DIR)
    WEIGHTS_DIR = "/kaggle/input/hubmap-vasculature-weights"   # used when TRAIN=False
    SEED = 42

    # ---- validation -----------------------------------------------------
    # "wsi"  : hold out WHOLE slides (all their tiles, dataset 1 and 2). The test set comes from other patients, so
    #          this is the honest cross-patient check. Fold k = k-th slide that has dataset-1 tiles (see the table
    #          printed in section 3). Slides without dataset-1 tiles are always used for training.
    # "block": old behaviour, spatial blocks inside the same slides (optimistic: the model has seen the patient).
    SPLIT_MODE = "wsi"
    N_FOLDS = 5              # only used when SPLIT_MODE == "block"
    SPLIT_BLOCK = 4096       # px, only used when SPLIT_MODE == "block"
    FOLDS_TO_RUN = [1]       # which held-out folds to train + validate
    FULL_SEEDS = []          # e.g. [0, 1]: ALSO train models on ALL labelled tiles (no validation, fixed epochs);
                             # saved as fold10_best.pth, fold11_best.pth ... so the inference notebook picks them up

    # ---- model / training -----------------------------------------------
    MODEL_SIZE = 1280        # tiles (512) are upscaled to this size inside the model (v1 used 1024)
    BATCH = 2                # lower this to 1 if you hit CUDA out-of-memory
    NUM_WORKERS = 2
    EPOCHS_S1 = 16           # stage 1: dataset 1 + dataset 2 (noisy, more data)
    EPOCHS_S2 = 6            # stage 2: dataset 1 only (clean) fine-tune
    LR_S1 = 1e-4
    LR_S2 = 2e-5
    WEIGHT_DECAY = 1e-2
    WARMUP_ITERS = 200
    EMA_DECAY = 0.998
    PATIENCE_S1 = 6          # early stopping (epochs without val-AP improvement)
    PATIENCE_S2 = 3
    LOSS_W = {"loss_mask": 2.0}   # everything else = 1.0
    TRAIN_EVAL_N = 64        # train tiles used to measure train-AP (over/under-fit diagnosis)

    # ---- colour / stain robustness (v2) -----------------------------------
    HED_P = 0.8              # probability of HED stain jitter per training tile
    HED_SIGMA = 0.2          # strength (calibrated: ~10 intensity levels mean change on tissue; raise for more)
    STAT_P = 0.5             # probability of colour-statistics transfer from another patient's tile
    STAT_STRENGTH = (0.2, 0.8)
    BANK_N = 400             # unlabelled dataset-3 tiles (9 other patients) used as colour references

    # ---- inference / metric ---------------------------------------------
    EVAL_IOU = 0.6           # IoU threshold of the competition metric (verify on the Evaluation tab)
    MIN_SCORE = 0.05
    MAX_DET = 100
    MASK_THR = 0.5
    TTA = ("id", "h", "v")   # identity, horizontal flip, vertical flip
    MS_SCALES = (0.8, 1.25)  # extra input sizes (x MODEL_SIZE) used by the "multi" TTA set
    MERGE_IOU = 0.55
    INF_BS = 2
    SAVE_EVERY = 25          # inference: rewrite submission.csv every N tiles


seed = CFG.SEED
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
if CFG.DEBUG:
    CFG.MODEL_SIZE, CFG.EPOCHS_S1, CFG.EPOCHS_S2, CFG.TRAIN_EVAL_N, CFG.BANK_N = 512, 1, 1, 8, 20

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./out")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def find_data_dir():
    hits = glob.glob("/kaggle/input/**/polygons.jsonl", recursive=True)
    if hits:
        return Path(hits[0]).parent
    for p in ("./data", "../input/hubmap-hacking-the-human-vasculature"):
        if (Path(p) / "polygons.jsonl").exists():
            return Path(p)
    raise FileNotFoundError("Could not find polygons.jsonl - add the competition data to the notebook.")


DATA_DIR = find_data_dir()
COLOR_BANK = np.zeros((0, 2, 3), np.float32)   # filled in section 3 (needs the data)
print("device:", DEVICE, "| AMP:", USE_AMP, "| data:", DATA_DIR)
print("torch", torch.__version__, "| torchvision", torchvision.__version__)

## 2. Utilities: annotation parsing, metric, TTA merging, RLE, colour augmentation

In [ ]:
# ---------------------------------------------------------------------------
# Pure numpy / OpenCV utilities: annotation parsing, metric, TTA merging, RLE
# (no torch needed here, so they can be unit-tested on their own)
# ---------------------------------------------------------------------------
import json, base64, zlib
import numpy as np
import cv2


def read_polygons(path):
    recs = {}
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            recs[r["id"]] = r["annotations"]
    return recs


def ann_to_mask(coords, h=512, w=512):
    """coords: list of rings; ring 0 = exterior, further rings = holes."""
    m = np.zeros((h, w), np.uint8)
    for k, ring in enumerate(coords):
        pts = np.round(np.asarray(ring, dtype=np.float64)).astype(np.int32).reshape(-1, 1, 2)
        if len(pts) < 3:
            continue
        cv2.fillPoly(m, [pts], 1 if k == 0 else 0)
    return m


def load_instances(annos, h=512, w=512, cls="blood_vessel", min_area=12):
    out = []
    for a in annos:
        if a["type"] != cls:
            continue
        m = ann_to_mask(a["coordinates"], h, w)
        if m.sum() >= min_area:
            out.append(m)
    return np.stack(out) if out else np.zeros((0, h, w), np.uint8)


def masks_to_boxes_np(masks):
    b = np.zeros((len(masks), 4), np.float32)
    for k, m in enumerate(masks):
        ys = np.flatnonzero(m.any(1)); xs = np.flatnonzero(m.any(0))
        if len(xs):
            b[k] = [xs[0], ys[0], xs[-1] + 1, ys[-1] + 1]
    return b


# ------------------------------- metric ------------------------------------
def mask_iou_matrix(pred, gt):
    """pred (N,H,W) bool, gt (M,H,W) bool -> (N,M) IoU."""
    if len(pred) == 0 or len(gt) == 0:
        return np.zeros((len(pred), len(gt)), np.float32)
    p = pred.reshape(len(pred), -1).astype(np.float32)
    g = gt.reshape(len(gt), -1).astype(np.float32)
    inter = p @ g.T
    union = p.sum(1)[:, None] + g.sum(1)[None] - inter
    return (inter / np.maximum(union, 1)).astype(np.float32)


def match_image(iou, scores, thr):
    """COCO-style greedy matching, highest score first. Returns (tp flags, sorted scores, matched IoU)."""
    order = np.argsort(-scores, kind="stable")
    tp = np.zeros(len(order), bool)
    miou = np.zeros(len(order), np.float32)
    taken = np.zeros(iou.shape[1], bool)
    if iou.shape[1]:
        for rank, pi in enumerate(order):
            cand = iou[pi].copy()
            cand[taken] = -1
            g = int(cand.argmax())
            if cand[g] >= thr:
                tp[rank] = True; taken[g] = True; miou[rank] = cand[g]
    return tp, scores[order], miou


def ap_from_matches(scores, tp, n_gt):
    """101-point interpolated AP (COCO style) for one class."""
    if n_gt == 0:
        return float("nan")
    if len(scores) == 0:
        return 0.0
    o = np.argsort(-scores, kind="stable")
    tp = tp[o].astype(np.float64)
    tpc, fpc = np.cumsum(tp), np.cumsum(1 - tp)
    rec, prec = tpc / n_gt, tpc / (tpc + fpc)
    prec = np.maximum.accumulate(prec[::-1])[::-1]
    idx = np.searchsorted(rec, np.linspace(0, 1, 101), side="left")
    q = np.array([prec[i] if i < len(prec) else 0.0 for i in idx])
    return float(q.mean())


def build_cache(iid, pred_masks, pred_scores, gt_masks):
    """Compact per-image record (IoU matrix instead of masks) so evaluation stays light on RAM."""
    gt_masks = gt_masks.astype(bool)
    return {
        "id": iid,
        "iou": mask_iou_matrix(pred_masks.astype(bool), gt_masks),
        "scores": np.asarray(pred_scores, np.float32),
        "gt_area": gt_masks.reshape(len(gt_masks), -1).sum(1) if len(gt_masks) else np.zeros(0),
        "n_gt": len(gt_masks),
    }


IOU_GRID = [round(t, 2) for t in np.arange(0.5, 0.951, 0.05)]


def evaluate_cache(caches, main_thr=0.6):
    thrs = sorted(set(IOU_GRID + [round(main_thr, 2)]))
    n_gt = int(sum(c["n_gt"] for c in caches))
    res = {"n_gt": n_gt, "n_pred": int(sum(len(c["scores"]) for c in caches)), "ap": {}}
    for t in thrs:
        S, T = [], []
        for c in caches:
            tp, s, mi = match_image(c["iou"], c["scores"], t)
            S.append(s); T.append(tp)
        S = np.concatenate(S) if S else np.zeros(0)
        T = np.concatenate(T) if T else np.zeros(0, bool)
        res["ap"][t] = ap_from_matches(S, T, n_gt)
        if t == round(main_thr, 2):
            res["S"], res["T"] = S, T
    res["main_ap"] = res["ap"][round(main_thr, 2)]
    res["map_5095"] = float(np.mean([res["ap"][t] for t in IOU_GRID]))
    per_img = []
    for c in caches:
        tp, s, _ = match_image(c["iou"], c["scores"], main_thr)
        per_img.append(ap_from_matches(s, tp, c["n_gt"]))
    res["per_image_ap"] = np.array(per_img)
    res["caches"] = caches
    return res


def instance_stats(res, score_thr):
    S, T, n_gt = res["S"], res["T"], res["n_gt"]
    keep = S >= score_thr
    tp = int((keep & T).sum()); fp = int((keep & ~T).sum()); fn = n_gt - tp
    p = tp / max(tp + fp, 1); r = tp / max(n_gt, 1)
    f1 = 2 * p * r / max(p + r, 1e-9)
    return {"tp": tp, "fp": fp, "fn": fn, "precision": p, "recall": r, "f1": f1,
            "instance_acc": tp / max(tp + fp + fn, 1)}


def f1_curve(res, grid=None):
    grid = np.linspace(0.05, 0.95, 91) if grid is None else grid
    rows = [dict(thr=t, **instance_stats(res, t)) for t in grid]
    return rows


# ------------------------- TTA / ensemble merging --------------------------
def _boxes_overlap(a, b):
    return not (a[2] <= b[0] or b[2] <= a[0] or a[3] <= b[1] or b[3] <= a[1])


def merge_predictions(members, iou_thr=0.55, mask_thr=0.5, min_score=0.05, max_det=100):
    """members: list of dicts {probs (N,H,W) float, boxes (N,4), scores (N,)} from TTA views / models.
    Instances from different members are clustered by mask IoU; masks are averaged and the score is the mean
    over *all* members (a member that missed the instance contributes 0)."""
    n_m = len(members)
    H, W = members[0]["probs"].shape[1:]
    items = [(float(m["scores"][k]), t, k) for t, m in enumerate(members) for k in range(len(m["scores"]))]
    items.sort(key=lambda x: -x[0])
    clusters = []
    for sc, t, k in items:
        m = members[t]
        box, prob = m["boxes"][k], m["probs"][k]
        pb = prob > mask_thr
        pa = pb.sum()
        best, best_iou = -1, 0.0
        for ci, c in enumerate(clusters):
            if t in c["members"] or not _boxes_overlap(box, c["box"]):
                continue
            inter = np.logical_and(pb, c["rep"]).sum()
            if inter == 0:
                continue
            iou = inter / (pa + c["rep_area"] - inter)
            if iou > best_iou:
                best_iou, best = iou, ci
        if best >= 0 and best_iou >= iou_thr:
            c = clusters[best]
            c["probs"].append(prob.astype(np.float32)); c["scores"].append(sc); c["members"].add(t)
        else:
            clusters.append({"rep": pb, "rep_area": pa, "box": box, "probs": [prob.astype(np.float32)],
                             "scores": [sc], "members": {t}})
    out = []
    for c in clusters:
        mk = np.mean(c["probs"], 0) > mask_thr
        out.append((sum(c["scores"]) / n_m, mk))
    out = [o for o in out if o[0] >= min_score and o[1].any()]
    out.sort(key=lambda x: -x[0])
    out = out[:max_det]
    masks = np.stack([o[1] for o in out]) if out else np.zeros((0, H, W), bool)
    return masks, np.array([o[0] for o in out], np.float32)


# ------------------------------ RLE encoding -------------------------------
def encode_mask(mask):
    """COCO RLE counts -> zlib -> base64 (the encoding I recall from the competition's Evaluation page;
    the notebook cross-checks it against sample_submission.csv before writing the final file)."""
    from pycocotools import mask as coco_mask
    m = np.asfortranarray(mask.astype(np.uint8)[:, :, None])
    counts = coco_mask.encode(m)[0]["counts"]
    return base64.b64encode(zlib.compress(counts, zlib.Z_BEST_COMPRESSION)).decode("ascii")


def decode_mask(s, h, w):
    from pycocotools import mask as coco_mask
    counts = zlib.decompress(base64.b64decode(s))
    return coco_mask.decode({"size": [h, w], "counts": counts})


def prediction_string(masks, scores, cls_id=0):
    return " ".join(f"{cls_id} {float(s):.5f} {encode_mask(m)}" for m, s in zip(masks, scores))


# ------------------------- colour / stain augmentation ----------------------
_RGB_FROM_HED = np.array([[0.65, 0.70, 0.29], [0.07, 0.99, 0.11], [0.27, 0.57, 0.78]], np.float32)
_HED_FROM_RGB = np.linalg.inv(_RGB_FROM_HED).astype(np.float32)
_LOG_ADJ = np.float32(np.log(1e-6))


def hed_jitter(img, rng, sigma=0.2, beta_scale=0.05):
    """HED stain jitter (after Tellez et al. 2018): split the tile into haematoxylin / eosin / DAB stain
    concentrations, randomly scale each (1 +- sigma) and shift it (+- sigma * beta_scale), and recombine.
    Only tissue pixels are changed (glass background stays white). Mimics patient / lab / scanner stain
    differences. (The published sigma=0.05 'light' setting is for a different value scale; the defaults here
    were calibrated so the average tissue colour change is moderate - see the notebook's augmentation plot.)"""
    rgb = np.maximum(img.astype(np.float32) / 255.0, 1e-6)
    hed = (np.log(rgb) / _LOG_ADJ).reshape(-1, 3) @ _HED_FROM_RGB
    hed = hed * rng.uniform(1 - sigma, 1 + sigma, 3).astype(np.float32) \
        + rng.uniform(-sigma, sigma, 3).astype(np.float32) * beta_scale
    out = np.exp((hed @ _RGB_FROM_HED) * _LOG_ADJ)
    out = np.clip(out * 255.0, 0, 255).reshape(img.shape).astype(np.uint8)
    tissue = img.mean(2) < 225
    return np.where(tissue[..., None], out, img)


def lab_stats(img):
    """(2,3) array: mean and std of L,a,b over tissue pixels (glass background excluded)."""
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB).astype(np.float32)
    m = lab[..., 0] < 225
    px = lab[m] if m.mean() > 0.02 else lab.reshape(-1, 3)
    return np.stack([px.mean(0), px.std(0) + 1e-3]).astype(np.float32)


def lab_transfer(img, ref, strength):
    """Reinhard-style colour transfer: move the tile's LAB mean/std towards `ref` (a lab_stats() entry from
    another patient's tile) by `strength` in [0, 1]. Glass background is left untouched."""
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB).astype(np.float32)
    m = lab[..., 0] < 225
    src = lab[m] if m.mean() > 0.02 else lab.reshape(-1, 3)
    mu_s, sd_s = src.mean(0), src.std(0) + 1e-3
    mu_t = mu_s + strength * (ref[0] - mu_s)
    sd_t = sd_s + strength * (ref[1] - sd_s)
    ratio = np.clip(sd_t / sd_s, 0.5, 2.0)
    out = (lab - mu_s) * ratio + mu_t
    out = np.where(m[..., None], out, lab)
    return cv2.cvtColor(np.clip(out, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB)

## 3. Load data, build folds, build the colour bank
In `wsi` mode a fold is one **whole held-out slide**. Look at the two tables printed here: they show which slides carry the clean dataset-1 labels and therefore which folds exist.

In [ ]:
from sklearn.model_selection import GroupKFold

tile_meta_path = sorted(DATA_DIR.glob("tile_meta*.csv"))[0]
meta = pd.read_csv(tile_meta_path)
ANN = read_polygons(DATA_DIR / "polygons.jsonl")
print(f"tile_meta: {len(meta):,} tiles | annotated ids: {len(ANN):,}")
print(meta.dataset.value_counts().sort_index().to_string())

# ---- labelled table: only blood_vessel instances are used ------------------
ds_of = dict(zip(meta.id, meta.dataset))
rows, AREAS = [], {1: [], 2: []}
for iid, annos in tqdm(ANN.items(), desc="parsing annotations"):
    ins = load_instances(annos)
    rows.append({"id": iid, "n_vessel": len(ins)})
    ds = int(ds_of.get(iid, 0))
    if ds in AREAS and len(ins):
        AREAS[ds].append(ins.reshape(len(ins), -1).sum(1))
AREAS = {k: (np.concatenate(v) if v else np.zeros(0)) for k, v in AREAS.items()}

lab = meta.merge(pd.DataFrame(rows), on="id", how="inner")
# Dataset-2 annotations are sparse: a tile with zero vessels there is more likely "not annotated" than "empty",
# so those tiles are dropped. Dataset-1 tiles (expert reviewed) keep their empty tiles as true negatives.
lab = lab[(lab.dataset == 1) | (lab.n_vessel > 0)].reset_index(drop=True)
lab["group"] = (lab.source_wsi.astype(str) + "_" + (lab.i // CFG.SPLIT_BLOCK).astype(str) + "_"
                + (lab.j // CFG.SPLIT_BLOCK).astype(str))
print("\nlabelled tiles per slide (source_wsi) and dataset:")
print(pd.crosstab(lab.dataset, lab.source_wsi).to_string())

if CFG.DEBUG:
    lab = pd.concat([lab[lab.dataset == 1].sample(min(60, (lab.dataset == 1).sum()), random_state=0),
                     lab[lab.dataset == 2].sample(min(60, (lab.dataset == 2).sum()), random_state=0)]).reset_index(drop=True)

lab["fold"] = -1
if CFG.SPLIT_MODE == "wsi":
    HELD = [int(w) for w in sorted(lab.loc[lab.dataset == 1, "source_wsi"].unique())]
    lab["fold"] = lab.source_wsi.map({w: k for k, w in enumerate(HELD)}).fillna(-1).astype(int)
    print(f"\nWSI split: fold k = one whole slide held out (dataset 1 AND 2 tiles). Slides with dataset-1 tiles: {HELD}")
    print("fold -1 = slides that never get held out (always training data)")
    assert all(0 <= f < len(HELD) for f in CFG.FOLDS_TO_RUN), f"FOLDS_TO_RUN must be in 0..{len(HELD) - 1} for SPLIT_MODE='wsi'"
else:   # spatial-block GroupKFold over ds1+ds2: tiles of a block never straddle train/val
    for f, (_, va) in enumerate(GroupKFold(n_splits=CFG.N_FOLDS).split(lab, groups=lab.group)):
        lab.loc[va, "fold"] = f
print("\nlabelled tiles per dataset / fold (validation always uses dataset 1 only):")
print(pd.crosstab(lab.dataset, lab.fold).to_string())

# ---- cache the labelled images in RAM (~1.3 GB for 1.6k tiles) -------------
def read_rgb(path):
    im = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if im is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

IMG = {iid: read_rgb(DATA_DIR / "train" / f"{iid}.tif") for iid in tqdm(lab.id, desc="loading images")}
print("image shape example:", next(iter(IMG.values())).shape)

# ---- colour bank: LAB colour statistics of unlabelled dataset-3 tiles (9 other patients). Only colours are used,
# never labels. Training tiles are sometimes shifted towards these colours so the model sees patient-to-patient
# stain variation it would otherwise never meet.
_ds3 = meta[meta.dataset == 3]
_bank_ids = _ds3.id.sample(min(CFG.BANK_N, len(_ds3)), random_state=CFG.SEED).tolist() if len(_ds3) else []
if _bank_ids:
    COLOR_BANK = np.stack([lab_stats(read_rgb(DATA_DIR / "train" / f"{i}.tif")) for i in tqdm(_bank_ids, desc="colour bank")])
print("colour bank:", COLOR_BANK.shape)

## 4. Explore the data

In [ ]:
# ---------------------------------------------------------------------------
# Plot style: one validated categorical order (blue, orange, aqua), one-hue sequential ramp,
# blue<->red diverging pair with a gray midpoint, hairline grid, no dual axes, legends for >= 2 series.
# ---------------------------------------------------------------------------
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

SURF, INK, INK2, MUTED, GRID, BASE = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE, AQUA, RED = "#2a78d6", "#eb6834", "#1baf7a", "#e34948"
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
DIV = LinearSegmentedColormap.from_list("div_br", [BLUE, "#f0efec", RED])

mpl.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF, "figure.dpi": 110,
    "axes.edgecolor": BASE, "axes.labelcolor": INK2, "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.labelsize": 10, "legend.frameon": False, "legend.fontsize": 9,
    "font.family": "sans-serif", "lines.linewidth": 2.0, "figure.autolayout": False,
})


def panel_title(ax, title, sub=None):
    ax.set_title(title, color=INK, loc="left", pad=16 if sub else 8)
    if sub:
        ax.text(0, 1.015, sub, transform=ax.transAxes, color=MUTED, fontsize=8.5, va="bottom")


def fig_axes(nrows, ncols, w, h, **kw):
    fig, axes = plt.subplots(nrows, ncols, figsize=(w, h), **kw)
    return fig, np.atleast_1d(axes)


DS_COL = {1: BLUE, 2: ORANGE, 3: AQUA}
DS_NAME = {1: "dataset 1 (expert)", 2: "dataset 2 (sparse)", 3: "dataset 3 (unlabelled)"}


def plot_data_overview():
    # --- 1. composition ------------------------------------------------------
    fig, ax = fig_axes(1, 3, 15, 3.8)
    comp = meta.groupby("dataset").size()
    ax[0].bar([DS_NAME[d].split(" (")[0] for d in comp.index], comp.values, color=BLUE, width=0.55)
    for x, v in enumerate(comp.values):
        ax[0].text(x, v, f"{v:,}", ha="center", va="bottom", color=INK2, fontsize=9)
    panel_title(ax[0], "Tiles per dataset", "all tile_meta rows"); ax[0].set_ylabel("tiles")
    ax[0].margins(y=0.15)

    n1, n2 = lab[lab.dataset == 1].n_vessel, lab[lab.dataset == 2].n_vessel
    bins = np.arange(0, max(n1.max(), n2.max(), 2) + 2)
    ax[1].hist(n1, bins=bins, density=True, color=BLUE, alpha=0.75, label=DS_NAME[1], rwidth=0.9)
    ax[1].hist(n2, bins=bins, density=True, color=ORANGE, alpha=0.65, label=DS_NAME[2], rwidth=0.9)
    panel_title(ax[1], "Vessels per tile", "density; dataset 2 tiles with 0 vessels were dropped")
    ax[1].set_xlabel("blood_vessel instances"); ax[1].legend()

    for d, c in ((1, BLUE), (2, ORANGE)):
        a = AREAS[d]
        if len(a):
            ax[2].hist(np.log10(a), bins=40, density=True, histtype="stepfilled", color=c, alpha=0.35)
            ax[2].hist(np.log10(a), bins=40, density=True, histtype="step", color=c, lw=1.8, label=DS_NAME[d])
    panel_title(ax[2], "Instance size", "mask area in pixels (log10)"); ax[2].set_xlabel("log10(area px)"); ax[2].legend()
    plt.tight_layout(); plt.show()

    # --- 2. tile maps (position of every tile inside each whole-slide image) ---
    wsis = sorted(meta.source_wsi.unique())
    nc = 5; nr = math.ceil(len(wsis) / nc)
    fig, axes = fig_axes(nr, nc, 15, 2.7 * nr)
    axes = axes.ravel()
    for ax, w in zip(axes, wsis):
        sub = meta[meta.source_wsi == w]
        for d in (3, 2, 1):
            s = sub[sub.dataset == d]
            ax.scatter(s.j, s.i, s=5, color=DS_COL[d], alpha=0.8, linewidths=0)
        ax.invert_yaxis(); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        ax.set_title(f"WSI {w}", fontsize=10, loc="left")
    for ax in axes[len(wsis):]:
        ax.axis("off")
    fig.legend(handles=[Line2D([], [], marker="o", ls="", color=DS_COL[d], label=DS_NAME[d]) for d in (1, 2, 3)],
               loc="lower right", ncol=3, bbox_to_anchor=(0.99, 0.0))
    fig.suptitle("Where every tile sits in its whole-slide image (maps)", x=0.01, ha="left", fontweight="bold")
    plt.tight_layout(rect=(0, 0.04, 1, 0.97)); plt.show()

    # --- 3. fold map: which spatial block lands in which validation fold ------
    fig, axes = fig_axes(1, 2, 13, 3.8)
    cnt = pd.crosstab(lab.source_wsi, lab.fold)
    im = axes[0].imshow(cnt.values, cmap=SEQ, aspect="auto")
    axes[0].set_xticks(range(cnt.shape[1])); axes[0].set_xticklabels(cnt.columns)
    axes[0].set_yticks(range(cnt.shape[0])); axes[0].set_yticklabels(cnt.index); axes[0].grid(False)
    for (r, c), v in np.ndenumerate(cnt.values):
        axes[0].text(c, r, v, ha="center", va="center", fontsize=8, color="white" if v > cnt.values.max() * 0.5 else INK)
    panel_title(axes[0], "Labelled tiles per WSI x fold", "spatial-block GroupKFold"); axes[0].set_xlabel("fold"); axes[0].set_ylabel("WSI")
    plt.colorbar(im, ax=axes[0], fraction=0.04)
    rr = lab.groupby("fold").agg(n=("id", "size"), ds1=("dataset", lambda s: (s == 1).sum()), vessels=("n_vessel", "sum"))
    axes[1].bar(rr.index.astype(str), rr.vessels, color=BLUE, width=0.55)
    panel_title(axes[1], "Vessel instances per fold", "should be roughly balanced"); axes[1].set_xlabel("fold")
    plt.tight_layout(); plt.show()


def contour_overlay(img, gt=None, pred=None, scores=None, min_score=0.3):
    out = img.copy()
    if gt is not None:
        cv2.drawContours(out, gt, -1, (27, 175, 122), 2)        # aqua = ground truth
    if pred is not None:
        keep = [c for c, s in zip(pred, scores) if s >= min_score]
        cv2.drawContours(out, [x for c in keep for x in c], -1, (235, 104, 52), 2)   # orange = prediction
    return out


def plot_samples(n=8, seed=0):
    rs = np.random.RandomState(seed)
    pick = list(rs.choice(lab[lab.dataset == 1].id, min(n // 2, (lab.dataset == 1).sum()), replace=False)) + \
           list(rs.choice(lab[lab.dataset == 2].id, min(n // 2, (lab.dataset == 2).sum()), replace=False))
    fig, axes = fig_axes(2, math.ceil(len(pick) / 2), 14, 7.4)
    for ax, iid in zip(axes.ravel(), pick):
        ins = load_instances(ANN[iid])
        cnts = [cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2] for m in ins]
        ax.imshow(contour_overlay(IMG[iid], [c for cs in cnts for c in cs])); ax.axis("off")
        ds = int(lab.loc[lab.id == iid, "dataset"].iloc[0])
        ax.set_title(f"ds{ds} | {len(ins)} vessels", fontsize=9, loc="left")
    fig.suptitle("Ground-truth blood vessels (aqua)  -  top row: dataset 1, bottom row: dataset 2", x=0.01, ha="left", fontweight="bold")
    plt.tight_layout(rect=(0, 0, 1, 0.96)); plt.show()


plot_data_overview()
plot_samples()

## 5. Dataset, augmentation, model, inference helpers

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


# ----------------------------- augmentation --------------------------------
def augment(img, masks, rng):
    """Geometry (flips, rot90, scale/rotate/shift) is applied jointly to the image and every instance mask;
    colour jitter (HSV, per-channel gain, blur) mimics stain / scanner variation between slides."""
    h, w = img.shape[:2]
    if rng.random() < 0.5:
        img = img[:, ::-1]; masks = masks[:, :, ::-1]
    if rng.random() < 0.5:
        img = img[::-1]; masks = masks[:, ::-1]
    k = int(rng.integers(4))
    img = np.ascontiguousarray(np.rot90(img, k)); masks = np.ascontiguousarray(np.rot90(masks, k, axes=(1, 2)))
    if rng.random() < 0.7:
        s, ang = rng.uniform(0.75, 1.35), rng.uniform(-25, 25)
        M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, s)
        M[:, 2] += rng.uniform(-0.1, 0.1, 2) * np.array([w, h])
        bg = tuple(int(x) for x in np.median(img.reshape(-1, 3), 0))
        img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=bg)
        if len(masks):
            masks = np.stack([cv2.warpAffine(m, M, (w, h), flags=cv2.INTER_NEAREST) for m in masks])
    return color_aug(img, rng), masks


def color_aug(img, rng):
    """Colour / stain augmentation. 1) shift the tile's colour statistics towards a random tile from another
    patient (colour bank), 2) HED stain jitter, 3) mild HSV + per-channel gain, 4) occasional blur."""
    if len(COLOR_BANK) and rng.random() < CFG.STAT_P:
        img = lab_transfer(img, COLOR_BANK[int(rng.integers(len(COLOR_BANK)))], rng.uniform(*CFG.STAT_STRENGTH))
    if rng.random() < CFG.HED_P:
        img = hed_jitter(img, rng, CFG.HED_SIGMA)
    if rng.random() < 0.6:
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[..., 0] = (hsv[..., 0] + rng.uniform(-4, 4)) % 180
        hsv[..., 1] *= rng.uniform(0.85, 1.15)
        hsv[..., 2] *= rng.uniform(0.9, 1.1)
        img = cv2.cvtColor(np.clip(hsv, 0, [179, 255, 255]).astype(np.uint8), cv2.COLOR_HSV2RGB)
    img = np.clip(img.astype(np.float32) * rng.uniform(0.95, 1.05, 3), 0, 255).astype(np.uint8)
    if rng.random() < 0.2:
        img = cv2.GaussianBlur(img, (0, 0), rng.uniform(0.5, 1.5))
    return img


def to_tensor(img):
    return torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1).float().div_(255.0)


class VesselDS(Dataset):
    def __init__(self, ids, train):
        self.ids, self.train, self._rng = list(ids), train, None

    def __len__(self):
        return len(self.ids)

    def rng(self):
        if self._rng is None:   # one generator per worker; torch's base seed changes every epoch
            self._rng = np.random.default_rng(torch.initial_seed() % (2 ** 32))
        return self._rng

    def __getitem__(self, idx):
        iid = self.ids[idx]
        img = IMG[iid]
        masks = load_instances(ANN[iid], *img.shape[:2])
        if self.train:
            img, masks = augment(img, masks, self.rng())
        keep = [k for k in range(len(masks)) if masks[k].sum() >= 12]
        masks = masks[keep] if len(masks) else masks
        boxes = masks_to_boxes_np(masks)
        ok = ((boxes[:, 2] - boxes[:, 0]) >= 2) & ((boxes[:, 3] - boxes[:, 1]) >= 2) if len(boxes) else np.zeros(0, bool)
        masks, boxes = masks[ok], boxes[ok]
        tgt = {"boxes": torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),
               "labels": torch.ones(len(boxes), dtype=torch.int64),
               "masks": torch.as_tensor(np.ascontiguousarray(masks), dtype=torch.uint8)}
        return to_tensor(img), tgt


def collate(batch):
    return tuple(zip(*batch))


# -------------------------------- model ------------------------------------
def build_model(pretrained=True):
    kw = dict(min_size=CFG.MODEL_SIZE, max_size=CFG.MODEL_SIZE, box_score_thresh=0.01,
              box_nms_thresh=0.5, box_detections_per_img=CFG.MAX_DET + 50)
    model = None
    if pretrained:
        try:   # needs Internet ON the first time (COCO weights); fine for a training run
            model = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT,
                                             trainable_backbone_layers=5, **kw)
        except Exception as e:
            print("!! could not download COCO weights -> training from scratch will be MUCH worse:", repr(e)[:150])
    if model is None:
        model = maskrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None, **kw)
    in_f = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_f, 2)
    in_m = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_m, 256, 2)
    return model


def freeze_bn(model):
    """Detection batches are tiny, so BatchNorm statistics stay frozen (affine params still train)."""
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()


class ModelEMA:
    def __init__(self, model, decay):
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)
        self.decay, self.n = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])


# ------------------------------- inference ---------------------------------
def _flip(x, mode):
    return x.flip(-1) if mode == "h" else x.flip(-2) if mode == "v" else x


def _unflip_boxes(b, mode, h, w):
    b = b.copy()
    if mode == "h":
        b[:, [0, 2]] = w - b[:, [2, 0]]
    elif mode == "v":
        b[:, [1, 3]] = h - b[:, [3, 1]]
    return b


def _unflip_masks(m, mode):
    return m[:, :, ::-1] if mode == "h" else m[:, ::-1] if mode == "v" else m


def view_sets():
    """Named TTA view sets. A view = (flip mode, model input size).
    single: 1 view | flips: identity + h + v flips | multi: flips + extra sizes (smaller and larger input)."""
    b = CFG.MODEL_SIZE
    lo, hi = (int(round(b * f / 32.0)) * 32 for f in CFG.MS_SCALES)
    flips = [(m, b) for m in CFG.TTA]
    return {"single": [("id", b)], "flips": flips, "multi": flips + [("id", lo), ("h", lo), ("id", hi), ("h", hi)]}


def _set_size(model, size):
    model.transform.min_size = (int(size),)
    model.transform.max_size = int(size)


@torch.no_grad()
def predict_members(models, imgs, views=(("id", None),), min_score=0.02, bs=None):
    """Runs every model x view; returns, per image, a list of member dicts for merge_predictions().
    `views`: list of (flip mode, input size) tuples (size None = CFG.MODEL_SIZE); plain mode strings also work.
    Masks and boxes always come back in the original 512 x 512 tile coordinates, whatever the input size."""
    bs = bs or CFG.INF_BS
    views = [(v, None) if isinstance(v, str) else tuple(v) for v in views]
    out = [[] for _ in imgs]
    for model in models:
        model.eval()
        try:
            for mode, size in views:
                size = int(size or CFG.MODEL_SIZE)
                _set_size(model, size)
                for s in range(0, len(imgs), bs):
                    chunk = imgs[s:s + bs]
                    x = [_flip(to_tensor(im), mode).to(DEVICE) for im in chunk]
                    with torch.autocast(device_type="cuda", enabled=USE_AMP):
                        res = model(x)
                    for j, r in enumerate(res):
                        keep = r["scores"] >= min_score
                        h, w = chunk[j].shape[:2]
                        probs = r["masks"][keep, 0].float().cpu().numpy().astype(np.float16)
                        out[s + j].append({
                            "mode": mode, "size": size,
                            "probs": np.ascontiguousarray(_unflip_masks(probs, mode)),
                            "boxes": _unflip_boxes(r["boxes"][keep].float().cpu().numpy(), mode, h, w),
                            "scores": r["scores"][keep].float().cpu().numpy()})
        finally:
            _set_size(model, CFG.MODEL_SIZE)     # always restore the training size
    return out


def eval_ids(model, ids, views=(("id", None),), mask_thr=None):
    """Streaming evaluation on a list of labelled tile ids -> evaluate_cache() result."""
    mask_thr = CFG.MASK_THR if mask_thr is None else mask_thr
    caches = []
    for s in range(0, len(ids), CFG.INF_BS):
        chunk = list(ids[s:s + CFG.INF_BS])
        mems = predict_members([model], [IMG[i] for i in chunk], views)
        for iid, mem in zip(chunk, mems):
            pm, ps = merge_predictions(mem, CFG.MERGE_IOU, mask_thr, CFG.MIN_SCORE, CFG.MAX_DET)
            caches.append(build_cache(iid, pm, ps, load_instances(ANN[iid])))
    return evaluate_cache(caches, CFG.EVAL_IOU)

## 5b. Check the colour augmentation before training
The augmented tiles should look like plausible other-patient slides. The second plot should show augmented tiles spreading towards the dataset-3 colour cloud.

In [ ]:
# ---------------------------------------------------------------------------
# Sanity-check the colour augmentation visually and numerically before spending GPU hours on it.
# ---------------------------------------------------------------------------
def plot_aug_examples(n=3, k=4, seed=1):
    rs = np.random.RandomState(seed)
    ids = list(rs.choice(lab.id.values, n, replace=False))
    fig, axes = fig_axes(n, k + 1, 3.0 * (k + 1), 3.1 * n, squeeze=False)
    for r, iid in enumerate(ids):
        axes[r, 0].imshow(IMG[iid]); axes[r, 0].set_title("original", fontsize=10, loc="left")
        for c in range(k):
            axes[r, c + 1].imshow(color_aug(IMG[iid], np.random.default_rng(100 * r + c)))
            axes[r, c + 1].set_title(f"augmented {c + 1}", fontsize=10, loc="left")
        for ax in axes[r]:
            ax.axis("off")
    fig.suptitle("Colour / stain augmentation (geometry not shown): should look like other patients' slides, not like noise",
                 x=0.01, ha="left", fontweight="bold")
    plt.tight_layout(rect=(0, 0, 1, 0.95)); plt.show()


def plot_stain_diversity(n=250, seed=0):
    """Mean tissue colour (CIELAB) per tile: labelled training slides vs unlabelled dataset-3 slides (other patients)
    vs the same training tiles after colour augmentation. Augmented tiles should spread towards the dataset-3 cloud."""
    rs = np.random.RandomState(seed)
    ids = list(rs.choice(list(IMG), min(n, len(IMG)), replace=False))
    orig = np.stack([lab_stats(IMG[i])[0] for i in ids])
    rng = np.random.default_rng(seed)
    aug = np.stack([lab_stats(color_aug(IMG[i], rng))[0] for i in ids])
    bank = COLOR_BANK[:, 0, :] if len(COLOR_BANK) else np.zeros((0, 3))
    groups = [("labelled tiles (slides 1-4)", orig, BLUE), ("dataset-3 tiles (other patients)", bank, ORANGE),
              ("labelled tiles after augmentation", aug, AQUA)]
    fig, ax = fig_axes(1, 2, 12, 4.4)
    for name, g, col in groups:
        if len(g):
            ax[0].scatter(g[:, 1], g[:, 2], s=14, color=col, alpha=0.6, linewidths=0, label=name)
            ax[1].scatter(g[:, 0], g[:, 1], s=14, color=col, alpha=0.6, linewidths=0, label=name)
    panel_title(ax[0], "Tissue colour per tile: a* vs b*", "each dot = one tile's mean colour"); ax[0].set_xlabel("mean a* (green-red)"); ax[0].set_ylabel("mean b* (blue-yellow)"); ax[0].legend()
    panel_title(ax[1], "Tissue brightness vs a*", "L* is lightness"); ax[1].set_xlabel("mean L*"); ax[1].set_ylabel("mean a*")
    plt.tight_layout(); plt.show()
    spread = lambda g: g[:, 1:].std(0).mean() if len(g) else float("nan")
    print(f"colour spread (std of a*,b*): labelled {spread(orig):.2f} | dataset 3 {spread(bank):.2f} | augmented {spread(aug):.2f}")


plot_aug_examples()
plot_stain_diversity()

## 6. Train
Two stages per model (dataset 1+2, then clean dataset 1). Held-out folds use early stopping on validation AP; `FULL_SEEDS` models train on all labelled tiles for the fixed number of epochs and keep the final EMA weights.

In [ ]:
def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except AttributeError:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def make_sched(opt, total, warm):
    def f(it):
        if it < warm:
            return (it + 1) / warm
        p = (it - warm) / max(1, total - warm)
        return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * p))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)


def weighted_loss(ld):
    return sum(CFG.LOSS_W.get(k, 1.0) * v for k, v in ld.items())


def make_loader(ids, train):
    return DataLoader(VesselDS(ids, train), batch_size=CFG.BATCH, shuffle=train, num_workers=CFG.NUM_WORKERS,
                      collate_fn=collate, drop_last=train and len(ids) > CFG.BATCH, pin_memory=USE_AMP)


def train_one_epoch(model, ema, loader, opt, sched, scaler, desc):
    model.train(); freeze_bn(model)
    tot, n = 0.0, 0
    for imgs, tgts in tqdm(loader, leave=False, desc=desc):
        imgs = [x.to(DEVICE, non_blocking=True) for x in imgs]
        tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
        with torch.autocast(device_type="cuda", enabled=USE_AMP):
            loss = weighted_loss(model(imgs, tgts))
        if not torch.isfinite(loss):
            opt.zero_grad(set_to_none=True); continue
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        scaler.step(opt); scaler.update(); sched.step(); ema.update(model)
        tot += loss.item(); n += 1
    return tot / max(n, 1)


@torch.no_grad()
def val_loss(model, loader):
    """Detection models only return losses in train mode; BN is frozen so this is a clean validation loss."""
    model.train(); freeze_bn(model)
    tot, n = 0.0, 0
    for imgs, tgts in loader:
        imgs = [x.to(DEVICE) for x in imgs]
        tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
        with torch.autocast(device_type="cuda", enabled=USE_AMP):
            tot += weighted_loss(model(imgs, tgts)).item(); n += 1
    model.eval()
    return tot / max(n, 1)


def run_fold(fold, full_k=None):
    """fold: held-out fold id (validated, early-stopped). full_k: train on ALL labelled tiles instead (no
    validation, fixed epochs, last EMA weights) and save as fold{10+full_k}_best.pth."""
    full = full_k is not None
    if full:
        name, tr, ids_va, seed = f"fold{10 + full_k}", lab, [], CFG.SEED + 1000 + full_k
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    else:
        name, tr, seed = f"fold{fold}", lab[lab.fold != fold], CFG.SEED + fold
        ids_va = lab[(lab.fold == fold) & (lab.dataset == 1)].id.tolist()
        assert len(ids_va) > 0, f"fold {fold} has no dataset-1 validation tiles"
    ids_s1 = tr.id.tolist()                               # dataset 1 + 2
    ids_s2 = tr[tr.dataset == 1].id.tolist()              # dataset 1 only (clean)
    rs = np.random.RandomState(seed)
    ids_tr_eval = list(rs.choice(ids_s2, min(CFG.TRAIN_EVAL_N, len(ids_s2)), replace=False))
    print(f"\n=== {name}{' (ALL data, no validation)' if full else ''}: stage1 {len(ids_s1)} tiles | "
          f"stage2 {len(ids_s2)} | validation {len(ids_va)} (dataset 1) ===")
    va_loader = make_loader(ids_va, train=False) if ids_va else None

    model = build_model(pretrained=True).to(DEVICE)
    hist, best = [], {"ap": -1.0, "state": None, "epoch": -1}
    ckpt = OUT_DIR / f"{name}_best.pth"
    for stage, (ids, epochs, lr, pat) in enumerate(
            [(ids_s1, CFG.EPOCHS_S1, CFG.LR_S1, CFG.PATIENCE_S1), (ids_s2, CFG.EPOCHS_S2, CFG.LR_S2, CFG.PATIENCE_S2)], 1):
        if stage == 2 and best["state"] is not None:
            model.load_state_dict(best["state"])          # fine-tune from the best stage-1 weights
        loader = make_loader(ids, train=True)
        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=CFG.WEIGHT_DECAY)
        sched = make_sched(opt, epochs * len(loader), CFG.WARMUP_ITERS if stage == 1 else 20)
        scaler, ema = make_scaler(), ModelEMA(model, CFG.EMA_DECAY)
        stage_best, bad = -1.0, 0
        for ep in range(1, epochs + 1):
            t0 = time.time()
            tl = train_one_epoch(model, ema, loader, opt, sched, scaler, f"s{stage} ep{ep}")
            flag = ""
            if full:
                vl = val_ap = tr_ap = float("nan")
                best.update(epoch=len(hist) + 1, state={k: v.detach().cpu().clone() for k, v in ema.ema.state_dict().items()})
            else:
                vl = val_loss(ema.ema, va_loader)
                val_ap = eval_ids(ema.ema, ids_va)["main_ap"]
                tr_ap = eval_ids(ema.ema, ids_tr_eval)["main_ap"]
                val_ap, tr_ap = (0.0 if np.isnan(v) else v for v in (val_ap, tr_ap))
                if val_ap > best["ap"]:
                    best.update(ap=val_ap, epoch=len(hist) + 1,
                                state={k: v.detach().cpu().clone() for k, v in ema.ema.state_dict().items()})
                    torch.save(best["state"], ckpt); flag = "  * best"
                if val_ap > stage_best + 1e-4:
                    stage_best, bad = val_ap, 0
                else:
                    bad += 1
            hist.append({"fold": name, "stage": stage, "epoch": len(hist) + 1, "train_loss": tl, "val_loss": vl,
                         "train_ap": tr_ap, "val_ap": val_ap, "lr": opt.param_groups[0]["lr"]})
            print(f"[{name} | stage {stage} | ep {ep:02d}] train_loss {tl:.3f} val_loss {vl:.3f} | "
                  f"train AP@{CFG.EVAL_IOU} {tr_ap:.3f} val AP@{CFG.EVAL_IOU} {val_ap:.3f} | {time.time()-t0:.0f}s{flag}")
            if not full and bad >= pat:
                print(f"   early stop (no val improvement for {pat} epochs)"); break
        del opt, sched, scaler, ema, loader; gc.collect(); torch.cuda.empty_cache()
    if full:
        torch.save(best["state"], ckpt)
    model.load_state_dict(best["state"])
    return model.eval(), pd.DataFrame(hist), ids_va, ids_tr_eval


FOLD_MODELS, FULL_MODELS, HIST, VAL_IDS, TRAIN_EVAL_IDS = {}, {}, [], {}, {}
if CFG.TRAIN:
    for f in CFG.FOLDS_TO_RUN:
        m, h, iv, it = run_fold(f)
        FOLD_MODELS[f], VAL_IDS[f], TRAIN_EVAL_IDS[f] = m, iv, it
        HIST.append(h)
    for k in CFG.FULL_SEEDS:
        m, h, _, _ = run_fold(None, full_k=k)
        FULL_MODELS[10 + k] = m
        HIST.append(h)
    HIST = pd.concat(HIST, ignore_index=True) if HIST else pd.DataFrame()
    HIST.to_csv(OUT_DIR / "history.csv", index=False)
    print("\nsaved:", sorted(p.name for p in OUT_DIR.glob("fold*_best.pth")))
else:
    wfiles = sorted(glob.glob(str(Path(CFG.WEIGHTS_DIR) / "**" / "fold*_best.pth"), recursive=True))
    assert wfiles, f"no fold*_best.pth under {CFG.WEIGHTS_DIR}"
    for wf in wfiles:
        f = int(Path(wf).stem.split("_")[0].replace("fold", ""))
        m = build_model(pretrained=False)
        m.load_state_dict(torch.load(wf, map_location="cpu")); FOLD_MODELS[f] = m.to(DEVICE).eval()
    print("loaded", len(FOLD_MODELS), "fold models for inference:", sorted(FOLD_MODELS))

## 7. Held-out evaluation, inference-setting sweep, over/under-fitting check

In [ ]:
# ---------------------------------------------------------------------------
# Held-out evaluation: every fold model is scored ONLY on its own held-out dataset-1 tiles
# (in SPLIT_MODE="wsi" these are tiles of a slide the model never saw = a cross-patient estimate).
# We sweep the TTA set (single view / flips / flips + multi-scale) and the mask threshold, and keep the best
# by AP@IoU 0.6. The expensive "multi" set must beat the best cheaper set by >= 0.005 AP to be chosen.
# ---------------------------------------------------------------------------
VIEWS = view_sets()
SWEEP = [("single", 0.4), ("single", 0.5), ("single", 0.6),
         ("flips", 0.4), ("flips", 0.5), ("flips", 0.6),
         ("multi", 0.5)]
VIS = {}          # per-image contours for the overlay plots (light on RAM)


def contours_of(masks):
    return [cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2] for m in masks]


def run_sweep(fold, model, ids):
    caches = {cfg: [] for cfg in SWEEP}
    for s in range(0, len(ids), CFG.INF_BS):
        chunk = list(ids[s:s + CFG.INF_BS])
        mems = predict_members([model], [IMG[i] for i in chunk], VIEWS["multi"])   # run all views once
        for iid, mem in zip(chunk, mems):
            gt = load_instances(ANN[iid]).astype(bool)
            for cfg in SWEEP:
                name, thr = cfg
                want = set(VIEWS[name])
                use = [m for m in mem if (m["mode"], m["size"]) in want]
                pm, ps = merge_predictions(use, CFG.MERGE_IOU, thr, CFG.MIN_SCORE, CFG.MAX_DET)
                caches[cfg].append(build_cache(iid, pm, ps, gt))
                if cfg == ("flips", 0.5):
                    VIS[iid] = {"pred": contours_of(pm), "scores": ps, "gt": contours_of(gt)}
    return caches


def pick_best(sweep_res, margin=0.005):
    cheap = [c for c in sweep_res if c[0] != "multi"]
    best_cheap = max(cheap, key=lambda c: sweep_res[c]["main_ap"])
    multi = [c for c in sweep_res if c[0] == "multi"]
    if multi:
        bm = max(multi, key=lambda c: sweep_res[c]["main_ap"])
        if sweep_res[bm]["main_ap"] >= sweep_res[best_cheap]["main_ap"] + margin:
            return bm
    return best_cheap


if CFG.TRAIN and FOLD_MODELS:
    ALL = {cfg: [] for cfg in SWEEP}
    FOLD_RES = {}
    for f, model in FOLD_MODELS.items():
        cs = run_sweep(f, model, VAL_IDS[f])
        for cfg in SWEEP:
            ALL[cfg] += cs[cfg]
        FOLD_RES[f] = evaluate_cache(cs[("flips", 0.5)], CFG.EVAL_IOU)
    SWEEP_RES = {cfg: evaluate_cache(ALL[cfg], CFG.EVAL_IOU) for cfg in SWEEP}
    BEST_CFG = pick_best(SWEEP_RES)
    BEST = SWEEP_RES[BEST_CFG]
    tbl = pd.DataFrame([{"views": c[0], "n_views": len(VIEWS[c[0]]), "mask_thr": c[1],
                         f"AP@{CFG.EVAL_IOU}": SWEEP_RES[c]["main_ap"], "mAP[.5:.95]": SWEEP_RES[c]["map_5095"]} for c in SWEEP])
    print("Inference-setting sweep (held-out tiles; n_views = relative inference cost per model):")
    print(tbl.round(4).to_string(index=False))
    print(f"\nchosen: views='{BEST_CFG[0]}', mask_thr={BEST_CFG[1]}  ->  AP@{CFG.EVAL_IOU} = {BEST['main_ap']:.4f}")
    print("(7 configs were compared on the same tiles, so expect a small optimistic bias)")
else:
    BEST_CFG = ("flips", 0.5)
    print("no validated fold models in this run: using views='flips', mask_thr=0.5")

# ------------------------------ metric table --------------------------------
if CFG.TRAIN and FOLD_MODELS:
    fc = f1_curve(BEST)
    best_f1 = max(fc, key=lambda r: r["f1"])
    METRICS = pd.Series({
        f"AP@{CFG.EVAL_IOU} (competition-style mAP)": BEST["main_ap"],
        "AP@0.5": BEST["ap"][0.5], "AP@0.75": BEST["ap"][0.75], "mAP@[0.5:0.95]": BEST["map_5095"],
        "mean per-image AP@0.6": float(np.nanmean(BEST["per_image_ap"])),
        "best-F1 score threshold": best_f1["thr"], "precision @ best F1": best_f1["precision"],
        "recall @ best F1": best_f1["recall"], "F1 @ best F1": best_f1["f1"],
        "instance accuracy TP/(TP+FP+FN)": best_f1["instance_acc"],
    })
    print("\nHeld-out validation metrics (dataset 1 only)")
    print(METRICS.round(4).to_string())
    if len(FOLD_RES) > 1:
        print("\nAP per fold:", {f: round(r["main_ap"], 4) for f, r in FOLD_RES.items()})

    # ------------- over / under-fitting diagnosis -----------------------------
    diag = []
    for f, model in FOLD_MODELS.items():
        tr = eval_ids(model, TRAIN_EVAL_IDS[f], views=VIEWS[BEST_CFG[0]], mask_thr=BEST_CFG[1])["main_ap"]
        diag.append({"fold": f, "train_AP": tr, "val_AP": FOLD_RES[f]["main_ap"]})
    DIAG = pd.DataFrame(diag); DIAG["gap"] = DIAG.train_AP - DIAG.val_AP
    print("\n", DIAG.round(3).to_string(index=False))
    gap = DIAG.gap.mean()
    if gap > 0.15:
        print(f"-> train-val gap {gap:.2f} > 0.15: the model does not transfer to unseen data well "
              f"(expected for held-out slides). Stronger augmentation / more varied data help; more epochs do not.")
    elif DIAG.train_AP.mean() < 0.5:
        print("-> train AP itself is low: UNDERFITTING. Try more epochs, higher MODEL_SIZE, a bigger backbone.")
    else:
        print(f"-> gap {gap:.2f} is small and train AP is healthy: no strong sign of over/under-fitting.")
    print("   (thresholds are rules of thumb; the curves below are the real evidence)")

    print("\n=== settings to copy into the inference notebook ===")
    print(f"   CFG.MODEL_SIZE = {CFG.MODEL_SIZE}   CFG.TTA_MODE = '{BEST_CFG[0]}'   CFG.MASK_THR = {BEST_CFG[1]}")

## 8. Result plots

In [ ]:
def plot_training(hist):
    for f, h in hist.groupby("fold"):
        if h.val_ap.isna().all():
            print(f"{f}: trained on all data (no validation) - final train loss {h.train_loss.iloc[-1]:.3f}")
            continue
        fig, ax = fig_axes(1, 3, 15, 3.8)
        x = h.epoch.values
        s2 = h.index[h.stage == 2]
        ax[0].plot(x, h.train_loss, color=BLUE, marker="o", ms=4, label="train loss")
        ax[0].plot(x, h.val_loss, color=ORANGE, marker="o", ms=4, label="val loss (EMA)")
        panel_title(ax[0], "Loss", "weighted sum of Mask R-CNN losses"); ax[0].set_xlabel("epoch"); ax[0].legend()
        ax[1].plot(x, h.train_ap, color=BLUE, marker="o", ms=4, label="train-subset AP")
        ax[1].plot(x, h.val_ap, color=ORANGE, marker="o", ms=4, label="validation AP")
        ax[1].fill_between(x, h.val_ap, h.train_ap, where=h.train_ap >= h.val_ap, color=BLUE, alpha=0.08)
        b = int(h.val_ap.values.argmax())
        ax[1].scatter([x[b]], [h.val_ap.values[b]], s=70, color=ORANGE, zorder=5, edgecolor=SURF, linewidth=2)
        ax[1].annotate(f"best {h.val_ap.values[b]:.3f}", (x[b], h.val_ap.values[b]), xytext=(6, -14),
                       textcoords="offset points", color=INK2, fontsize=9)
        panel_title(ax[1], f"AP@{CFG.EVAL_IOU}", "shaded gap = generalisation gap"); ax[1].set_xlabel("epoch"); ax[1].legend()
        ax[2].plot(x, h.lr, color=AQUA, marker="o", ms=3)
        ax[2].set_yscale("log"); panel_title(ax[2], "Learning rate", "value at end of each epoch"); ax[2].set_xlabel("epoch")
        for a in ax:
            if len(s2):
                a.axvline(h.epoch[s2[0]] - 0.5, color=MUTED, ls=":", lw=1.2)
                a.text(h.epoch[s2[0]] - 0.4, a.get_ylim()[1], " stage 2: clean ds1", color=MUTED, fontsize=8, va="top")
        fig.suptitle(f"Training curves - {f}", x=0.01, ha="left", fontweight="bold")
        plt.tight_layout(rect=(0, 0, 1, 0.95)); plt.show()


def pr_arrays(res):
    S, T, n = res["S"], res["T"], res["n_gt"]
    o = np.argsort(-S, kind="stable"); tp = T[o].astype(float)
    tpc, fpc = np.cumsum(tp), np.cumsum(1 - tp)
    rec, prec = tpc / max(n, 1), tpc / np.maximum(tpc + fpc, 1e-9)
    return rec, prec, np.maximum.accumulate(prec[::-1])[::-1]


def plot_metric_dashboard(best, sweep_res, best_cfg, diag):
    fc = pd.DataFrame(f1_curve(best))
    bf = fc.loc[fc.f1.idxmax()]
    # --- row 1: AP vs IoU, PR curve, P/R/F1 vs score threshold ------------------
    fig, ax = fig_axes(1, 3, 15, 4.2)
    ts = sorted(best["ap"]); vals = [best["ap"][t] for t in ts]
    cols = [ORANGE if abs(t - CFG.EVAL_IOU) < 1e-6 else BLUE for t in ts]
    ax[0].bar([f"{t:.2f}" for t in ts], vals, color=cols, width=0.62)
    for k, v in enumerate(vals):
        ax[0].text(k, v + 0.01, f"{v:.2f}", ha="center", fontsize=8, color=INK2)
    panel_title(ax[0], "AP across mask-IoU thresholds", f"orange = IoU {CFG.EVAL_IOU} (metric); mAP[.5:.95] = {best['map_5095']:.3f}")
    ax[0].set_xlabel("IoU threshold"); ax[0].set_ylim(0, 1.05); ax[0].tick_params(axis="x", labelsize=8)
    rec, prec, env = pr_arrays(best)
    ax[1].plot(rec, prec, color=MUTED, lw=0.8, label="raw precision")
    ax[1].plot(rec, env, color=BLUE, label="interpolated precision")
    ax[1].fill_between(rec, env, color=BLUE, alpha=0.12)
    panel_title(ax[1], f"Precision-recall @ IoU {CFG.EVAL_IOU}", f"area under blue curve = AP {best['main_ap']:.3f}")
    ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].set_xlim(0, 1); ax[1].set_ylim(0, 1.02); ax[1].legend(loc="lower left")
    ax[2].plot(fc.thr, fc.precision, color=BLUE, label="precision")
    ax[2].plot(fc.thr, fc.recall, color=ORANGE, label="recall")
    ax[2].plot(fc.thr, fc.f1, color=AQUA, label="F1")
    ax[2].axvline(bf.thr, color=MUTED, ls=":", lw=1.2)
    panel_title(ax[2], "Instance metrics vs score threshold", f"best F1 {bf.f1:.3f} at score >= {bf.thr:.2f}")
    ax[2].set_xlabel("confidence threshold"); ax[2].set_ylim(0, 1.02); ax[2].legend(loc="lower center", ncol=3)
    plt.tight_layout(); plt.show()

    # --- row 2: config sweep, generalisation gap, per-image AP ------------------
    fig, ax = fig_axes(1, 3, 15, 4.0)
    names = [f"{t}\nmask>{m}" for t, m in sweep_res]
    v = [sweep_res[c]["main_ap"] for c in sweep_res]
    ax[0].bar(names, v, color=[ORANGE if c == best_cfg else BLUE for c in sweep_res], width=0.6)
    lo = max(0, min(v) - 0.05)
    for k, y in enumerate(v):
        ax[0].text(k, y + 0.002, f"{y:.3f}", ha="center", fontsize=8, color=INK2)
    ax[0].set_ylim(lo, max(v) + 0.03); ax[0].tick_params(axis="x", labelsize=7.5)
    panel_title(ax[0], "Inference settings", "view set and mask threshold (orange = chosen); y truncated")
    xs = np.arange(len(diag)); w = 0.36
    ax[1].bar(xs - w / 2, diag.train_AP, w, color=BLUE, label="train subset")
    ax[1].bar(xs + w / 2, diag.val_AP, w, color=ORANGE, label="held-out validation")
    for k, (a, b) in enumerate(zip(diag.train_AP, diag.val_AP)):
        ax[1].text(k - w / 2, a + 0.01, f"{a:.2f}", ha="center", fontsize=8, color=INK2)
        ax[1].text(k + w / 2, b + 0.01, f"{b:.2f}", ha="center", fontsize=8, color=INK2)
    ax[1].set_xticks(xs); ax[1].set_xticklabels([f"fold {f}" for f in diag.fold]); ax[1].set_ylim(0, 1.05)
    panel_title(ax[1], "Over/under-fitting check", "big train>>val gap = overfit; both low = underfit"); ax[1].legend()
    pia = best["per_image_ap"][~np.isnan(best["per_image_ap"])]
    ax[2].hist(pia, bins=np.linspace(0, 1, 21), color=BLUE, rwidth=0.9)
    ax[2].axvline(np.mean(pia), color=ORANGE, lw=1.8)
    ax[2].text(np.mean(pia), ax[2].get_ylim()[1] * 0.95, f" mean {np.mean(pia):.2f}", color=INK2, fontsize=9, va="top")
    panel_title(ax[2], "Per-tile AP", "how evenly the model performs across tiles"); ax[2].set_xlabel("AP@IoU 0.6 of one tile"); ax[2].set_ylabel("tiles")
    plt.tight_layout(); plt.show()

    # --- row 3: recall heatmap, count scatter, TP vs FP scores -----------------
    fig, ax = fig_axes(1, 3, 15, 4.2)
    edges = [0, 300, 1000, 3000, 10000, 1e9]; labels = ["<300", "300-1k", "1k-3k", "3k-10k", ">10k"]
    ious = [0.5, 0.6, 0.7, 0.8, 0.9]
    best_iou, areas = [], []
    for c in best["caches"]:
        if c["n_gt"] == 0:
            continue
        keep = c["scores"] >= bf.thr
        best_iou.append(c["iou"][keep].max(0) if keep.any() else np.zeros(c["n_gt"]))
        areas.append(c["gt_area"])
    best_iou, areas = np.concatenate(best_iou), np.concatenate(areas)
    H = np.full((len(labels), len(ious)), np.nan)
    for r in range(len(labels)):
        sel = (areas >= edges[r]) & (areas < edges[r + 1])
        if sel.sum():
            H[r] = [(best_iou[sel] >= t).mean() for t in ious]
    im = ax[0].imshow(H, cmap=SEQ, vmin=0, vmax=1, aspect="auto"); ax[0].grid(False)
    ax[0].set_xticks(range(len(ious))); ax[0].set_xticklabels([f">={t}" for t in ious])
    ax[0].set_yticks(range(len(labels))); ax[0].set_yticklabels(labels)
    for (r, c_), val in np.ndenumerate(H):
        if not np.isnan(val):
            ax[0].text(c_, r, f"{val:.2f}", ha="center", va="center", fontsize=9, color="white" if val > 0.5 else INK)
    panel_title(ax[0], "Recall by vessel size x IoU", f"at score >= {bf.thr:.2f}"); ax[0].set_xlabel("mask IoU"); ax[0].set_ylabel("GT area (px)")
    plt.colorbar(im, ax=ax[0], fraction=0.045)
    ng = np.array([c["n_gt"] for c in best["caches"]]); npd = np.array([(c["scores"] >= bf.thr).sum() for c in best["caches"]])
    jit = np.random.RandomState(0).uniform(-0.18, 0.18, len(ng))
    m = max(ng.max(), npd.max(), 1) + 1
    ax[1].plot([0, m], [0, m], color=MUTED, ls=":", lw=1.2)
    ax[1].scatter(ng + jit, npd + jit, s=22, color=BLUE, alpha=0.6, linewidths=0)
    panel_title(ax[1], "Vessels per tile: predicted vs true", "above the line = over-counting"); ax[1].set_xlabel("ground-truth count"); ax[1].set_ylabel("predicted count")
    S, T = best["S"], best["T"]
    bins = np.linspace(0, 1, 26)
    ax[2].hist(S[T], bins=bins, color=BLUE, alpha=0.75, label=f"true positives ({int(T.sum())})")
    ax[2].hist(S[~T], bins=bins, color=ORANGE, alpha=0.6, label=f"false positives ({int((~T).sum())})")
    ax[2].set_yscale("log"); ax[2].legend()
    panel_title(ax[2], "Confidence of TP vs FP", "good separation = well-ranked scores"); ax[2].set_xlabel("score")
    plt.tight_layout(); plt.show()


def plot_spatial_ap(best, lab_df):
    info = lab_df.set_index("id")
    ids = [c["id"] for c in best["caches"]]
    d = pd.DataFrame({"id": ids, "ap": best["per_image_ap"]}).join(info[["source_wsi", "i", "j"]], on="id").dropna()
    wsis = sorted(d.source_wsi.unique())
    if not len(wsis):
        return
    nc = min(4, len(wsis)); nr = math.ceil(len(wsis) / nc)
    fig, axes = fig_axes(nr, nc, 3.8 * nc + 1.2, 3.6 * nr, squeeze=False)
    axes = axes.ravel()
    for ax, w in zip(axes, wsis):
        s = d[d.source_wsi == w]
        sc = ax.scatter(s.j, s.i, c=s.ap, cmap=SEQ, vmin=0, vmax=1, s=80, marker="s", edgecolor=SURF, linewidth=1)
        ax.invert_yaxis(); ax.set_aspect("equal", adjustable="datalim"); ax.grid(False)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"WSI {w} | mean AP {s.ap.mean():.2f}", fontsize=10, loc="left")
    for ax in axes[len(wsis):]:
        ax.axis("off")
    fig.colorbar(sc, ax=axes.tolist(), fraction=0.025, label="tile AP")
    fig.suptitle("Where on the slide does the model do well? (validation tiles, AP map)", x=0.01, ha="left", fontweight="bold")
    plt.show()


def plot_overlays(best, vis, img_dict, k=4):
    pairs = [(a, c["id"]) for a, c in zip(best["per_image_ap"], best["caches"]) if not np.isnan(a) and c["id"] in vis]
    pairs.sort(key=lambda x: -x[0])
    chosen = [("best", p) for p in pairs[:k]] + [("worst", p) for p in pairs[-k:][::-1]]
    fig, axes = fig_axes(2, k, 3.6 * k, 7.4)
    for ax, (tag, (a, iid)) in zip(axes.ravel(), chosen):
        v = vis[iid]
        ax.imshow(_overlay_nested(img_dict[iid], v)); ax.axis("off")
        ax.set_title(f"{tag}: AP {a:.2f}", fontsize=10, loc="left")
    fig.legend(handles=[Line2D([], [], color=AQUA, lw=2, label="ground truth"),
                        Line2D([], [], color=ORANGE, lw=2, label="prediction (score >= 0.3)")],
               loc="lower right", ncol=2)
    fig.suptitle("Predictions vs ground truth - best tiles (top) and worst tiles (bottom)", x=0.01, ha="left", fontweight="bold")
    plt.tight_layout(rect=(0, 0.03, 1, 0.96)); plt.show()


def _overlay_nested(img, v, min_score=0.3):
    out = img.copy()
    cv2.drawContours(out, [c for cs in v["gt"] for c in cs], -1, (27, 175, 122), 2)
    keep = [cs for cs, s in zip(v["pred"], v["scores"]) if s >= min_score]
    cv2.drawContours(out, [c for cs in keep for c in cs], -1, (235, 104, 52), 2)
    return out


if CFG.TRAIN and len(HIST):
    plot_training(HIST)
if CFG.TRAIN and FOLD_MODELS:
    plot_metric_dashboard(BEST, SWEEP_RES, BEST_CFG, DIAG)
    plot_spatial_ap(BEST, lab)
    plot_overlays(BEST, VIS, IMG)

## 9. Predict the test set and write `submission.csv`
Convenience only. For the real submission use the lean inference notebook (it loads no training data).

In [ ]:
INFER_VIEWS_NAME, INFER_THR = BEST_CFG
models = [FOLD_MODELS[k] for k in sorted(FOLD_MODELS)] + [FULL_MODELS[k] for k in sorted(FULL_MODELS)]
# ---------------------------------------------------------------------------
# Robust test-set inference. Each tile is handled on its own:
#   attempt 1: all models x the chosen view set  ->  attempt 2: all models, one view  ->  attempt 3: first model only
#   if every attempt fails the tile gets an EMPTY prediction and the error is printed, so one bad tile
#   can never crash the whole submission. submission.csv is rewritten every CFG.SAVE_EVERY tiles.
# ---------------------------------------------------------------------------
test_files = sorted(glob.glob(str(DATA_DIR / "test" / "*.tif")))
ss_path = DATA_DIR / "sample_submission.csv"
sample = pd.read_csv(ss_path) if ss_path.exists() else None
VIEWS = view_sets()
assert len(models) > 0, "no models loaded"
COLS = ["id", "height", "width", "prediction_string"]
print(f"{len(test_files)} test tiles | {len(models)} models | first attempt: '{INFER_VIEWS_NAME}' = {len(VIEWS[INFER_VIEWS_NAME])} views per model "
      f"| mask_thr {INFER_THR} | input size {CFG.MODEL_SIZE}")

ATTEMPTS = [("full", None, VIEWS[INFER_VIEWS_NAME]), ("single-view", None, VIEWS["single"]), ("single-model", 1, VIEWS["single"])]
rows, how_count, errors = {}, {}, []


def run_tile(im, models_, modes):
    mem = predict_members(models_, [im], modes)[0]
    pm, ps = merge_predictions(mem, CFG.MERGE_IOU, INFER_THR, CFG.MIN_SCORE, CFG.MAX_DET)
    return prediction_string(pm, ps, cls_id=0), len(ps)


def write_csv():
    df = pd.DataFrame(list(rows.values()), columns=COLS)
    if sample is not None and set(sample.columns) == set(COLS):
        df = df[list(sample.columns)]
    tmp = OUT_DIR / "submission.csv.tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, OUT_DIR / "submission.csv")
    return df


t0 = time.time()
for n, path in enumerate(tqdm(test_files, desc="inference")):
    tid, h, w, im, text, how = Path(path).stem, 512, 512, None, "", "empty"
    try:
        im = read_rgb(path); h, w = im.shape[:2]
    except Exception as e:
        errors.append((tid, "read", repr(e)[:200])); print(f"[{tid}] cannot read image: {e!r}", flush=True)
    if im is not None:
        for name, k, modes in ATTEMPTS:
            try:
                text, _ = run_tile(im, models if k is None else models[:k], modes)
                how = name
                break
            except Exception as e:
                errors.append((tid, name, repr(e)[:200]))
                print(f"[{tid}] attempt '{name}' failed: {type(e).__name__}: {str(e)[:200]}", flush=True)
                traceback.print_exc(limit=4)
                gc.collect(); torch.cuda.empty_cache()
    how_count[how] = how_count.get(how, 0) + 1
    rows[tid] = {"id": tid, "height": h, "width": w, "prediction_string": text}
    if (n + 1) % CFG.SAVE_EVERY == 0:
        write_csv(); gc.collect()
        print(f"  {n + 1}/{len(test_files)} tiles | {(time.time() - t0) / (n + 1):.1f}s per tile | outcomes {how_count}", flush=True)

# any id listed in sample_submission.csv but not found as a file gets an empty row
if sample is not None:
    for _, r in sample.iterrows():
        if str(r["id"]) not in rows:
            rows[str(r["id"])] = {"id": str(r["id"]), "height": int(r["height"]), "width": int(r["width"]), "prediction_string": ""}
            print("added empty row for id missing from test folder:", r["id"])
sub = write_csv()

# format cross-check against the sample file
if sample is not None and "prediction_string" in sample.columns:
    ps0 = sample.iloc[0]["prediction_string"]
    if isinstance(ps0, str) and len(ps0.split()) >= 3:
        try:
            _ = decode_mask(ps0.split()[2], int(sample.iloc[0]["height"]), int(sample.iloc[0]["width"]))
            print("encoding check OK against sample_submission.csv")
        except Exception as e:
            print("!! sample prediction_string does NOT decode with my RLE scheme:", repr(e)[:150])

n_inst = sub.prediction_string.fillna("").map(lambda s: len(s.split()) // 3)
print(f"\nwrote {OUT_DIR / 'submission.csv'}: {len(sub)} rows, {int(n_inst.sum())} instances, "
      f"{int((n_inst == 0).sum())} empty tiles | outcomes {how_count} | {len(errors)} errors | {time.time() - t0:.0f}s total")
for e in errors[:20]:
    print("  error:", e)
sub.head()